In [0]:
%run ./utility/logger

In [0]:
dbutils.widgets.text('catalog',"")
dbutils.widgets.text('schema',"")
dbutils.widgets.text('env',"")

In [0]:
catalog = dbutils.widgets.get('catalog')
schema = dbutils.widgets.get('schema')
env = dbutils.widgets.get('env')

In [0]:
print(schema)

In [0]:
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW product_incremental AS

SELECT *
FROM commerce_stage_{env}.silver.product_stage

WHERE updated_ts >
(
    SELECT COALESCE(MAX(last_processed_ts), TIMESTAMP('1900-01-01'))
    FROM commerce_main_{env}.util.etl_control
    WHERE table_name = 'dim_product'
)
""")

In [0]:
%sql
select * from product_incremental

In [0]:
spark.sql(f"""
MERGE INTO {catalog}.{schema}.dim_product tgt

USING product_incremental src

ON tgt.product_id = src.product_id
AND tgt.is_current = true

WHEN MATCHED
AND
(
       tgt.product_name IS DISTINCT FROM src.product_name
    OR tgt.category IS DISTINCT FROM src.category
    OR tgt.brand IS DISTINCT FROM src.brand
)

THEN UPDATE SET

    tgt.is_current = false,
    tgt.effective_to = current_timestamp(),
    tgt.updated_ts = current_timestamp()
""")

In [0]:
spark.sql(f"""
INSERT INTO {catalog}.{schema}.dim_product
(
    product_id,
    product_name,
    category,
    brand,

    effective_from,
    effective_to,
    is_current,

    created_ts,
    updated_ts
)

SELECT

    src.product_id,
    src.product_name,
    src.category,
    src.brand,

    current_timestamp(),
    TIMESTAMP('9999-12-31 23:59:59'),
    true,

    current_timestamp(),
    current_timestamp()

FROM product_incremental src

LEFT JOIN {catalog}.{schema}.dim_product tgt

ON src.product_id = tgt.product_id
AND tgt.is_current = true

WHERE tgt.product_id IS NULL
""")

In [0]:
spark.sql(f"""
MERGE INTO commerce_main_{env}.util.etl_control tgt

USING
(
    SELECT
        'dim_product' AS table_name,
        MAX(updated_ts) AS last_processed_ts

    FROM commerce_stage_{env}.silver.product_stage
) src

ON tgt.table_name = src.table_name

WHEN MATCHED THEN
UPDATE SET
    tgt.last_processed_ts = src.last_processed_ts

WHEN NOT MATCHED THEN
INSERT
(
    table_name,
    last_processed_ts
)
VALUES
(
    src.table_name,
    src.last_processed_ts
)
""")